# Embedded VAE Inference on Zynq DPU

Welcome to this demonstration! In this notebook, we'll see how we can drastically accelerate Neural Network execution and reduce power consumption by offloading computations from a traditional embedded CPU to a dedicated FPGA hardware accelerator (the DPU).

This demonstration specifically highlights a **Variational Autoencoder (VAE)**. The encoder part of the network takes a high-resolution input image and compresses it down to a simple **6-value scalar vector**.

Let's import our tools and get started.

In [ ]:
import vae_demo_utils as utils
import warnings
warnings.filterwarnings('ignore')

### 1. Load Dataset and Hardware & Model Initialization
Here we load our test dataset images, we initialize the embedded CPU Pytorch model, load the DPU hardware overlay onto the FPGA fabric, and set up the board's physical power recorder.

In [ ]:
# 1. Load up to 1000 images for our performance benchmark
dataloader, dataset = utils.get_dataloader(img_path='dataset', num_samples=1000)
print(f"Loaded {len(dataset)} images for testing.")

# 2. Initialize models and hardware overlays
print("Initializing CPU Model...")
cpu_model = utils.setup_cpu_model('pre_trained_w_encoder.pt')

print("Initializing hardware overlay for DPU...")
overlay, dpu_runner, dpu_input, dpu_output = utils.setup_dpu_model(
    bitstream_path="../vitisai_bitstream/dpu.bit",
    model_path="zcu104_vaemodel1.xmodel"
)

# 3. Initialize hardware Power Recorder
print("Setting up hardware Power Recorder...")
recorder = utils.setup_power_recorder()

### 2. Information Compression & CPU Baseline
The original input image contains nearly 100,000 pixel values. Despite this complexity, the VAE encoder compresses all of this structure and feature information down into just **6 scalar numbers**.

In this step, we run the **embedded CPU baseline implementation** through our test dataset. It will record execution time and power. While it inferences the dataset, we will periodically display the input image alongside this 6-value latent representation that the model has compressed it into.

In [ ]:
# 3. Run the CPU Baseline and visualize during execution
if recorder: recorder.record(0.01) # Start capturing power telemetry

print("\nRunning inference on the embedded CPU...")

# Passing plot_every=50 tells the function to animate the plot directly during runtime!
cpu_latents, cpu_avg_time, cpu_total = utils.run_cpu_inference(
    cpu_model, dataloader, recorder, plot_every=50
)

# 4. Finish
print(f"-> CPU Execution completed in {cpu_total:.2f} seconds.")

# Note: The VAE model architecture technically computes the standard deviation as well, 
# but these first 6 values are what encode the structural meaning of the image.

### 3. FPGA Acceleration (DPU Inference)
Now we will stream the exact same workload into the dedicated **DPU (Data Processing Unit)** on the FPGA fabric. The `dpu_runner` driver handles allocating the tensors, transferring the data over the internal bridges, and computing the layers asynchronously.

In [ ]:
# 5. Run the DPU (FPGA) Model Acceleration
print("Running inference on the DPU hardware accelerator...")

dpu_latents, dpu_avg_time, dpu_total = utils.run_dpu_inference(
    dpu_runner, 
    dpu_input, 
    dpu_output, 
    dataloader, 
    recorder
)
print(f"-> DPU Execution completed in {dpu_total:.2f} seconds.")

# Stop power capture to seal the data frames
if recorder: recorder.stop()

### 4. Results Summary
Let's look at the average time required to process a single image. We will also plot the overall power consumption profile, directly extracting readings from the Zynq board's power rail.

In [ ]:
# Print full inference speedup and energy efficiency numbers
utils.print_performance_summary(cpu_avg_time, dpu_avg_time, recorder)

# Plot the captured power footprint over time
utils.plot_power_consumption(recorder)